In [11]:
# Librerias
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [3]:
data = pd.read_csv(r'C:\Users\Maria Paula\OneDrive\Documents\GitHub\Reto_Topo\DataFrameBabyFaceValidacion.csv')
df = pd.DataFrame(data)

In [7]:
DATA_CANDIDATES = [
    Path(r"C:\Users\Maria Paula\OneDrive\Documents\GitHub\Reto_Topo\DataFrameBabyFaceValidacion.csv"),
    Path("/mnt/data/DataFrameBabyFaceValidacion.xlsx"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontró DataFrameBabyFaceValidacion.xlsx. Coloca el Excel en la misma carpeta del notebook.")

OUTPUT_DIR = Path("babyface_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

MISSING_TOKENS = {'', '-', 'NA', 'NA ', 'N/A', 'nan', 'NaN', 'FIT_FAILED', 'FIT FAILED', 'FIND_FAILED', 'S/I', None}

def parse_time_to_seconds(x):
    """Convierte tiempos a segundos. Maneja segundos, milisegundos y formatos tipo HH:MM:SS.mmm."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        value = float(x)
        # Algunos registros están en milisegundos: 16433 -> 16.433 s.
        return value / 1000 if value > 1000 else value
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    if ':' in s:
        try:
            parts = [p.strip() for p in s.split(':')]
            if len(parts) == 3:
                h, m, sec = parts
                return float(h) * 3600 + float(m) * 60 + float(sec)
            if len(parts) == 2:
                m, sec = parts
                return float(m) * 60 + float(sec)
        except Exception:
            pass
    # Casos como 1.00.60 o 1,34,033: se interpretan como minutos, segundos y fracción.
    if re.fullmatch(r'\d+[\.,]\d+[\.,]\d+', s):
        nums = [int(p) for p in re.split(r'[\.,]', s)]
        if len(nums) == 3:
            minutes, seconds, frac = nums
            return minutes * 60 + seconds + frac / (10 ** len(str(frac)))
    try:
        value = float(s.replace(',', '.'))
        return value / 1000 if value > 1000 else value
    except Exception:
        return np.nan

def parse_numeric(x):
    """Convierte a float y manda tokens de error/missing a NaN."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    try:
        return float(s.replace(',', '.'))
    except Exception:
        return np.nan

def parse_day_number(x):
    """Extrae el número de día de etiquetas mixtas como 7E, 7P, 1E 2, 4.2."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    match = re.search(r'\d+(?:\.\d+)?', s)
    return float(match.group(0)) if match else np.nan

def parse_phase(x):
    """Extrae si la etiqueta de día tiene E, P, ambas o ninguna."""
    if pd.isna(x):
        return "sin_fase"
    s = str(x).strip().upper()
    if s in MISSING_TOKENS:
        return "sin_fase"
    has_e = 'E' in s
    has_p = 'P' in s
    if has_e and has_p:
        return "E/P"
    if has_e:
        return "E"
    if has_p:
        return "P"
    return "sin_fase"

def load_and_clean(path=DATA_PATH):
    raw = pd.read_csv(path)
    df = raw.copy()

    # Normalización de comentarios y día.
    df['comentarios'] = df['comentarios'].replace({'NA': np.nan, 'NA ': np.nan, '-': np.nan})
    df['dia_original'] = df['dia'].astype(str)
    df['dia_num'] = df['dia'].apply(parse_day_number)
    df['fase_dia'] = df['dia'].apply(parse_phase)

    time_cols = ['t_total', 't_prueba1', 't_prueba2']
    binary_cols = ['rechaza', 'llora', 'toca', 'prueba', 'consume']
    va_cols = [c for c in df.columns if c.startswith('valence') or c.startswith('arousal')]

    for c in time_cols:
        df[c] = df[c].apply(parse_time_to_seconds)
    for c in binary_cols:
        df[c] = df[c].apply(parse_numeric)
    for c in va_cols:
        df[c] = df[c].apply(parse_numeric)

    valence_cols = [c for c in df.columns if c.startswith('valence')]
    arousal_cols = [c for c in df.columns if c.startswith('arousal')]

    # Variables derivadas para EDA y TDA.
    df['valence_mean'] = df[valence_cols].mean(axis=1, skipna=True)
    df['arousal_mean'] = df[arousal_cols].mean(axis=1, skipna=True)
    df['valence_delta'] = df['valence_final'] - df['valence_inicio']
    df['arousal_delta'] = df['arousal_final'] - df['arousal_inicio']
    df['valence_p1_delta'] = df['valence_p1_after'] - df['valence_p1_before']
    df['valence_p2_delta'] = df['valence_p2_after'] - df['valence_p2_before']
    df['arousal_p1_delta'] = df['arousal_p1_after'] - df['arousal_p1_before']
    df['arousal_p2_delta'] = df['arousal_p2_after'] - df['arousal_p2_before']
    df['aceptacion_score'] = df[['prueba','consume']].mean(axis=1, skipna=True) - df[['rechaza','llora']].mean(axis=1, skipna=True)
    df['va_missing_count'] = df[valence_cols + arousal_cols].isna().sum(axis=1)
    return raw, df

raw, df_limpio = load_and_clean()
print(f"Archivo cargado: {DATA_PATH}")
print(f"Dimensiones raw: {raw.shape[0]} filas x {raw.shape[1]} columnas")
print(f"Dimensiones cleaned: {df.shape[0]} filas x {df.shape[1]} columnas")

Archivo cargado: C:\Users\Maria Paula\OneDrive\Documents\GitHub\Reto_Topo\DataFrameBabyFaceValidacion.csv
Dimensiones raw: 366 filas x 27 columnas
Dimensiones cleaned: 366 filas x 27 columnas


In [10]:
comentarios = df_limpio['comentarios']
comentarios.isna().sum()

np.int64(112)